In [2]:
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import copy
import pathlib
from pathlib import Path

In [3]:
DATASET_PATH = Path("../../../../datasets/historical")

def load_city_data(city_name):
    """
    Loads weather and AQI data for a city.

    Example:
        weather, aqi = load_city_data("Delhi")
    """

    weather_path = DATASET_PATH / f"{city_name}_weather_15july.csv"
    aqi_path = DATASET_PATH / f"{city_name}_AQI_15July.csv"

    weather = pd.read_csv(weather_path,skiprows=2)
    aqi = pd.read_csv(aqi_path,skiprows=2)

    print(f"\n{city_name}")
    print("-" * 40)
    print("Weather Shape :", weather.shape)
    print("AQI Shape     :", aqi.shape)

    return weather, aqi

In [4]:
Delhi_weather, Delhi_aqi = load_city_data("Delhi")

print(Delhi_weather.columns.tolist())
print(Delhi_aqi.columns.tolist())


Delhi
----------------------------------------
Weather Shape : (2208, 8)
AQI Shape     : (2210, 7)
['time', 'temperature_2m (°C)', 'relative_humidity_2m (%)', 'rain (mm)', 'surface_pressure (hPa)', 'cloud_cover (%)', 'wind_speed_10m (m/s)', 'wind_direction_10m (°)']
['time', 'pm10 (μg/m³)', 'pm2_5 (μg/m³)', 'carbon_monoxide (μg/m³)', 'nitrogen_dioxide (μg/m³)', 'sulphur_dioxide (μg/m³)', 'ozone (μg/m³)']


In [5]:
Delhi_aqi.head()

,time,pm10 (μg/m³),pm2_5 (μg/m³),carbon_monoxide (μg/m³),nitrogen_dioxide (μg/m³),sulphur_dioxide (μg/m³),ozone (μg/m³)
0,2026-07-19T18:30,661.2,137.4,293.0,29.2,17.1,80.0
1,time,pm10 (μg/m³),pm2_5 (μg/m³),nitrogen_dioxide (μg/m³),sulphur_dioxide (μg/m³),ozone (μg/m³),carbon_monoxide (μg/m³)
2,2026-04-15T00:00,323.2,60.3,19.1,14.2,85.0,555.0
3,2026-04-15T01:00,207.7,46.6,17.2,13.8,87.0,584.0
4,2026-04-15T02:00,129.5,36.9,16.5,14.3,89.0,616.0


In [6]:
Mumbai_weather , Mumbai_aqi = load_city_data("Mumbai")

print(Mumbai_weather.columns.tolist())
print(Mumbai_aqi.columns.tolist())


Mumbai
----------------------------------------
Weather Shape : (2208, 8)
AQI Shape     : (2210, 7)
['time', 'temperature_2m (°C)', 'relative_humidity_2m (%)', 'rain (mm)', 'surface_pressure (hPa)', 'cloud_cover (%)', 'wind_speed_10m (m/s)', 'wind_direction_10m (°)']
['time', 'pm10 (μg/m³)', 'pm2_5 (μg/m³)', 'carbon_monoxide (μg/m³)', 'nitrogen_dioxide (μg/m³)', 'sulphur_dioxide (μg/m³)', 'ozone (μg/m³)']


In [7]:
BLR_weather , BLR_aqi = load_city_data("Bengaluru")

print(BLR_weather.columns.tolist())
print(BLR_aqi.columns.tolist())


Bengaluru
----------------------------------------
Weather Shape : (2208, 8)
AQI Shape     : (2210, 7)
['time', 'temperature_2m (°C)', 'relative_humidity_2m (%)', 'rain (mm)', 'surface_pressure (hPa)', 'cloud_cover (%)', 'wind_speed_10m (m/s)', 'wind_direction_10m (°)']
['time', 'pm10 (μg/m³)', 'pm2_5 (μg/m³)', 'carbon_monoxide (μg/m³)', 'nitrogen_dioxide (μg/m³)', 'sulphur_dioxide (μg/m³)', 'ozone (μg/m³)']


In [8]:
def clean_weather(weather_df):
    """
    Cleans Open-Meteo weather data.

    Returns:
        Cleaned weather dataframe
    """

    weather = weather_df.copy()

    # Rename columns

    weather.rename(columns={
        "time": "Datetime",
        "temperature_2m (°C)": "Temperature",
        "relative_humidity_2m (%)": "Humidity",
        "rain (mm)": "Rain",
        "surface_pressure (hPa)": "Pressure",
        "cloud_cover (%)": "CloudCover",
        "wind_speed_10m (m/s)": "WindSpeed",
        "wind_direction_10m (°)": "WindDirection"
    }, inplace=True)

   
    weather["Datetime"] = pd.to_datetime(weather["Datetime"])
  
    # Sort
  
    weather.sort_values("Datetime", inplace=True)

    # Remove duplicates
   
    weather.drop_duplicates(subset="Datetime", inplace=True)

    # Missing values

    print("\nMissing Values (Weather)")
    print(weather.isnull().sum())

    # Interpolate numeric columns
    numeric_cols = weather.select_dtypes(include=np.number).columns

    weather[numeric_cols] = weather[numeric_cols].interpolate(
        method="linear"
    )

    # If any remain
    weather.bfill(inplace=True)
    weather.ffill(inplace=True)

    # Reset index
    
    weather.reset_index(drop=True, inplace=True)

    return weather

In [9]:
Delhi_weather_cleaned = clean_weather(Delhi_weather)
Delhi_weather_cleaned.head()


Missing Values (Weather)
Datetime         0
Temperature      0
Humidity         0
Rain             0
Pressure         0
CloudCover       0
WindSpeed        0
WindDirection    0
dtype: int64


,Datetime,Temperature,Humidity,Rain,Pressure,CloudCover,WindSpeed,WindDirection
0,2026-04-15 00:00:00,27.0,30,0.0,980.2,29,1.06,8
1,2026-04-15 01:00:00,26.2,31,0.0,979.5,0,1.21,7
2,2026-04-15 02:00:00,25.1,33,0.0,978.8,0,1.27,351
3,2026-04-15 03:00:00,24.4,35,0.0,978.6,0,1.41,337
4,2026-04-15 04:00:00,23.8,36,0.0,978.5,0,1.34,333


In [10]:
Mumbai_weather_cleaned = clean_weather(Mumbai_weather)
Mumbai_weather_cleaned.head()


Missing Values (Weather)
Datetime         0
Temperature      0
Humidity         0
Rain             0
Pressure         0
CloudCover       0
WindSpeed        0
WindDirection    0
dtype: int64


,Datetime,Temperature,Humidity,Rain,Pressure,CloudCover,WindSpeed,WindDirection
0,2026-04-15 00:00:00,27.5,79,0.0,1007.7,0,2.16,316
1,2026-04-15 01:00:00,27.2,80,0.0,1007.1,0,1.88,318
2,2026-04-15 02:00:00,26.3,84,0.0,1006.8,0,0.92,331
3,2026-04-15 03:00:00,25.9,87,0.0,1006.3,0,0.95,342
4,2026-04-15 04:00:00,25.8,88,0.0,1006.2,0,1.25,355


In [11]:
BLR_weather_cleaned = clean_weather(BLR_weather)
BLR_weather_cleaned.head()


Missing Values (Weather)
Datetime         0
Temperature      0
Humidity         0
Rain             0
Pressure         0
CloudCover       0
WindSpeed        0
WindDirection    0
dtype: int64


,Datetime,Temperature,Humidity,Rain,Pressure,CloudCover,WindSpeed,WindDirection
0,2026-04-15 00:00:00,26.3,41,0.0,911.4,0,2.48,137
1,2026-04-15 01:00:00,25.5,43,0.0,910.7,0,2.06,154
2,2026-04-15 02:00:00,24.8,45,0.0,910.2,0,2.10,179
3,2026-04-15 03:00:00,24.2,44,0.0,909.7,2,1.95,183
4,2026-04-15 04:00:00,23.7,44,0.0,909.9,0,2.04,205


In [12]:
def clean_aqi(df, city_name):

    df = df.copy()

    # 1. Remove duplicate header row
    
    df = df[df["time"] != "time"]

    # 2. Convert datetime


    df["time"] = pd.to_datetime(
        df["time"],
        errors="coerce"
    )

    # Remove rows where time conversion failed

    df = df.dropna(
        subset=["time"]
    )

    # 3. Convert pollutant columns

    pollutant_cols = [
        "pm10 (μg/m³)",
        "pm2_5 (μg/m³)",
        "carbon_monoxide (μg/m³)",
        "nitrogen_dioxide (μg/m³)",
        "sulphur_dioxide (μg/m³)",
        "ozone (μg/m³)"
    ]


    for col in pollutant_cols:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

    # 4. Rename columns

    df.rename(
        columns={
            "time": "Datetime",

            "pm10 (μg/m³)": "PM10",

            "pm2_5 (μg/m³)": "PM25",

            "carbon_monoxide (μg/m³)": "CO",

            "nitrogen_dioxide (μg/m³)": "NO2",

            "sulphur_dioxide (μg/m³)": "SO2",

            "ozone (μg/m³)": "O3"
        },
        inplace=True
    )

    # 5. Add city

    df["City"] = city_name

    # 6. Handle missing values

    pollution_features = [
        "PM10",
        "PM25",
        "CO",
        "NO2",
        "SO2",
        "O3"
    ]


    df[pollution_features] = (
        df[pollution_features]
        .ffill()
        .bfill()
    )

    # 7. Sort by datetime

    df.sort_values(
        "Datetime",
        inplace=True
    )


    # Reset index

    df.reset_index(
        drop=True,
        inplace=True
    )


    return df

In [13]:
Delhi_aqi_cleaned = clean_aqi(
    Delhi_aqi,
    "Delhi"
)

In [14]:
Mumbai_AQI_cleaned = clean_aqi(
    Mumbai_aqi,
    "Mumbai"
)

In [15]:
BLR_AQI_cleaned = clean_aqi(
    BLR_aqi,
    "Bengaluru"
)

### Merge weather data with that of AQI data of city 

In [16]:
Delhi_merged = pd.merge(
    Delhi_weather_cleaned,
    Delhi_aqi_cleaned,
    on="Datetime",
    how="inner"
)
Delhi_merged.head()

,Datetime,Temperature,Humidity,Rain,Pressure,CloudCover,WindSpeed,WindDirection,PM10,PM25,CO,NO2,SO2,O3,City
0,2026-04-15 00:00:00,27.0,30,0.0,980.2,29,1.06,8,323.2,60.3,19.1,14.2,85.0,555.0,Delhi
1,2026-04-15 01:00:00,26.2,31,0.0,979.5,0,1.21,7,207.7,46.6,17.2,13.8,87.0,584.0,Delhi
2,2026-04-15 02:00:00,25.1,33,0.0,978.8,0,1.27,351,129.5,36.9,16.5,14.3,89.0,616.0,Delhi
3,2026-04-15 03:00:00,24.4,35,0.0,978.6,0,1.41,337,101.3,32.4,17.4,16.0,87.0,639.0,Delhi
4,2026-04-15 04:00:00,23.8,36,0.0,978.5,0,1.34,333,94.1,31.2,19.6,18.6,84.0,665.0,Delhi


In [17]:
Mumbai_merged = pd.merge(
    Mumbai_weather_cleaned,
    Mumbai_AQI_cleaned,
    on="Datetime",
    how="inner"
)
Mumbai_merged.head()

,Datetime,Temperature,Humidity,Rain,Pressure,CloudCover,WindSpeed,WindDirection,PM10,PM25,CO,NO2,SO2,O3,City
0,2026-04-15 00:00:00,27.5,79,0.0,1007.7,0,2.16,316,64.3,23.5,14.1,10.9,78.0,316.0,Mumbai
1,2026-04-15 01:00:00,27.2,80,0.0,1007.1,0,1.88,318,64.0,23.6,14.3,10.7,77.0,342.0,Mumbai
2,2026-04-15 02:00:00,26.3,84,0.0,1006.8,0,0.92,331,62.9,23.5,14.7,10.7,76.0,378.0,Mumbai
3,2026-04-15 03:00:00,25.9,87,0.0,1006.3,0,0.95,342,61.2,23.5,15.2,10.5,75.0,432.0,Mumbai
4,2026-04-15 04:00:00,25.8,88,0.0,1006.2,0,1.25,355,60.1,23.7,15.9,10.4,73.0,497.0,Mumbai


In [18]:
BLR_merged = pd.merge(
    BLR_weather_cleaned,
    BLR_AQI_cleaned,
    on="Datetime",
    how="inner"
)
BLR_merged.head()

,Datetime,Temperature,Humidity,Rain,Pressure,CloudCover,WindSpeed,WindDirection,PM10,PM25,CO,NO2,SO2,O3,City
0,2026-04-15 00:00:00,26.3,41,0.0,911.4,0,2.48,137,19.5,17.0,19.4,10.0,89.0,394.0,Bengaluru
1,2026-04-15 01:00:00,25.5,43,0.0,910.7,0,2.06,154,20.1,17.3,18.2,10.5,89.0,347.0,Bengaluru
2,2026-04-15 02:00:00,24.8,45,0.0,910.2,0,2.10,179,21.2,18.0,18.1,11.1,86.0,322.0,Bengaluru
3,2026-04-15 03:00:00,24.2,44,0.0,909.7,2,1.95,183,22.6,19.2,20.8,11.9,77.0,332.0,Bengaluru
4,2026-04-15 04:00:00,23.7,44,0.0,909.9,0,2.04,205,24.3,20.6,24.8,12.8,65.0,365.0,Bengaluru


### Feature Engineering

In [19]:
def feature_engineering(df):

    df = df.copy()

    # Sort by time

    df = df.sort_values(
        "Datetime"
    )

    # Time Features

    df["hour"] = (
        df["Datetime"]
        .dt.hour
    )


    df["day"] = (
        df["Datetime"]
        .dt.day
    )


    df["month"] = (
        df["Datetime"]
        .dt.month
    )


    df["day_of_week"] = (
        df["Datetime"]
        .dt.dayofweek
    )

    # PM2.5 Lag Features

    lag_hours = [
        1,
        3,
        6,
        12,
        24
    ]


    for lag in lag_hours:

        df[f"PM25_lag_{lag}"] = (
            df["PM25"]
            .shift(lag)
        )

    # PM10 Lag Features

    for lag in lag_hours:

        df[f"PM10_lag_{lag}"] = (
            df["PM10"]
            .shift(lag)
        )

    # Rolling Mean

    df["PM25_roll_6"] = (
        df["PM25"]
        .rolling(
            window=6
        )
        .mean()
    )


    df["PM25_roll_24"] = (
        df["PM25"]
        .rolling(
            window=24
        )
        .mean()
    )



    df["PM10_roll_6"] = (
        df["PM10"]
        .rolling(
            window=6
        )
        .mean()
    )

    # Remove NaN

    df.dropna(
        inplace=True
    )

    # Reset index

    df.reset_index(
        drop=True,
        inplace=True
    )

    return df

In [20]:
Delhi_final = feature_engineering(
    Delhi_merged
)

In [21]:
Mumbai_final = feature_engineering(
    Mumbai_merged
)

In [22]:
BLR_final = feature_engineering(
    BLR_merged
)

In [23]:
BLR_merged.shape

(2208, 15)

In [24]:
BLR_final.shape

(2184, 32)

### Merge data of all cities

In [25]:
final_dataset = pd.concat(
    [
        Delhi_final,
        Mumbai_final,
        BLR_final
    ],
    ignore_index=True
)

In [26]:
final_dataset.shape

(6552, 32)

In [27]:
print(final_dataset.columns)

Index(['Datetime', 'Temperature', 'Humidity', 'Rain', 'Pressure', 'CloudCover',
       'WindSpeed', 'WindDirection', 'PM10', 'PM25', 'CO', 'NO2', 'SO2', 'O3',
       'City', 'hour', 'day', 'month', 'day_of_week', 'PM25_lag_1',
       'PM25_lag_3', 'PM25_lag_6', 'PM25_lag_12', 'PM25_lag_24', 'PM10_lag_1',
       'PM10_lag_3', 'PM10_lag_6', 'PM10_lag_12', 'PM10_lag_24', 'PM25_roll_6',
       'PM25_roll_24', 'PM10_roll_6'],
      dtype='str')


In [28]:
final_dataset.head()

,Datetime,Temperature,Humidity,Rain,Pressure,CloudCover,WindSpeed,WindDirection,PM10,PM25,...,PM25_lag_12,PM25_lag_24,PM10_lag_1,PM10_lag_3,PM10_lag_6,PM10_lag_12,PM10_lag_24,PM25_roll_6,PM25_roll_24,PM10_roll_6
0,2026-04-16 00:00:00,27.1,28,0.0,979.1,4,0.88,313,603.5,108.3,...,34.3,60.3,633.3,618.4,521.5,276.6,323.2,97.933333,55.512500,598.216667
1,2026-04-16 01:00:00,26.1,31,0.0,978.4,27,1.00,354,575.6,108.3,...,40.3,46.6,603.5,642.6,523.4,364.1,207.7,103.533333,58.083333,606.916667
2,2026-04-16 02:00:00,25.4,36,0.0,978.2,0,0.91,6,551.3,109.0,...,46.1,36.9,575.6,633.3,568.1,438.7,129.5,107.000000,61.087500,604.116667
3,2026-04-16 03:00:00,24.8,38,0.0,978.1,16,1.04,17,523.4,108.5,...,51.2,32.4,551.3,603.5,618.4,502.9,101.3,108.433333,64.258333,588.283333
4,2026-04-16 04:00:00,24.6,38,0.0,978.3,47,0.65,337,499.2,107.8,...,56.3,31.2,523.4,575.6,642.6,556.9,94.1,108.550000,67.450000,564.383333


In [29]:
final_dataset = final_dataset.sort_values(
    [
        "City",
        "Datetime"
    ]
)


final_dataset.reset_index(
    drop=True,
    inplace=True
)

### Create Targets

#### 24 hour shift

In [30]:
final_dataset["Target_PM25"] = (
    final_dataset
    .groupby("City")["PM25"]
    .shift(-24)
)

In [31]:
final_dataset["Target_PM10"] = (
    final_dataset
    .groupby("City")["PM10"]
    .shift(-24)
)

In [32]:
final_dataset[
    [
        "City",
        "Datetime",
        "PM25",
        "Target_PM25",
        "PM10",
        "Target_PM10"
    ]
].tail(30)

,City,Datetime,PM25,Target_PM25,PM10,Target_PM10
6522,Mumbai,2026-07-14 18:00:00,20.5,20.5,48.1,47.7
6523,Mumbai,2026-07-14 19:00:00,20.0,18.9,48.2,42.0
6524,Mumbai,2026-07-14 20:00:00,19.9,17.6,48.3,38.0
6525,Mumbai,2026-07-14 21:00:00,19.6,16.8,48.4,36.0
6526,Mumbai,2026-07-14 22:00:00,19.0,16.8,45.6,37.0
6527,Mumbai,2026-07-14 23:00:00,18.3,17.2,43.1,37.4
6528,Mumbai,2026-07-15 00:00:00,17.8,NaN,40.7,NaN
6529,Mumbai,2026-07-15 01:00:00,17.0,NaN,38.3,NaN
6530,Mumbai,2026-07-15 02:00:00,16.9,NaN,38.8,NaN
6531,Mumbai,2026-07-15 03:00:00,17.5,NaN,39.8,NaN


In [33]:
final_dataset.columns.tolist()

['Datetime',
 'Temperature',
 'Humidity',
 'Rain',
 'Pressure',
 'CloudCover',
 'WindSpeed',
 'WindDirection',
 'PM10',
 'PM25',
 'CO',
 'NO2',
 'SO2',
 'O3',
 'City',
 'hour',
 'day',
 'month',
 'day_of_week',
 'PM25_lag_1',
 'PM25_lag_3',
 'PM25_lag_6',
 'PM25_lag_12',
 'PM25_lag_24',
 'PM10_lag_1',
 'PM10_lag_3',
 'PM10_lag_6',
 'PM10_lag_12',
 'PM10_lag_24',
 'PM25_roll_6',
 'PM25_roll_24',
 'PM10_roll_6',
 'Target_PM25',
 'Target_PM10']

#### Remove rows without target

In [34]:
final_dataset.dropna(
    inplace=True
)

In [35]:
final_dataset.shape

(6480, 34)

#### Remove current PM2.5 and PM10

In [36]:
X = final_dataset.drop(
    columns=[
        "Datetime",
        "PM25",
        "PM10",
        "Target_PM25",
        "Target_PM10"
    ]
)


y = final_dataset[
    [
        "Target_PM25",
        "Target_PM10"
    ]
]

#### Encode City 

In [37]:
X = pd.get_dummies(
    X,
    columns=["City"],
    dtype=int
)

In [38]:
final_dataset.groupby("City").size()

City
Bengaluru    2160
Delhi        2160
Mumbai       2160
dtype: int64

### Train-Test Split 

In [39]:
from xgboost import XGBRegressor

In [40]:
train_parts = []
test_parts = []

for city, group in final_dataset.groupby("City"):
    group = group.sort_values("Datetime")

    split = int(len(group) * 0.8)

    train_parts.append(group.iloc[:split])
    test_parts.append(group.iloc[split:])

train_df = pd.concat(train_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

In [42]:
test_df.to_csv("test_dataset.csv", index=False)

In [46]:
X_train = train_df.drop(
    columns=[
        "Datetime",
        "PM25",
        "PM10",
        "Target_PM25",
        "Target_PM10"
    ]
)

y_train = train_df[
    [
        "Target_PM25",
        "Target_PM10"
    ]
]

X_test = test_df.drop(
    columns=[
        "Datetime",
        "PM25",
        "PM10",
        "Target_PM25",
        "Target_PM10"
    ]
)

y_test = test_df[
    [
        "Target_PM25",
        "Target_PM10"
    ]
]

#### Encode city columns

In [47]:
X_train = pd.get_dummies(
    X_train,
    columns=["City"],
    dtype=int
)

X_test = pd.get_dummies(
    X_test,
    columns=["City"],
    dtype=int
)

In [48]:
X_train, X_test = X_train.align(
    X_test,
    join="left",
    axis=1,
    fill_value=0
)

## PM2.5 Model

In [49]:
model_pm25 = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)


model_pm25.fit(
    X_train,
    y_train["Target_PM25"]
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


## PM10 Model

In [50]:
model_pm10 = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42
)


model_pm10.fit(
    X_train,
    y_train["Target_PM10"]
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


## Prediction

In [51]:
pred_pm25 = model_pm25.predict(
    X_test
)


pred_pm10 = model_pm10.predict(
    X_test
)

In [52]:
correction_factor = (
    y_test["Target_PM25"].mean()
    /
    pred_pm25.mean()
)

adjusted_pred_pm25 = (
    pred_pm25 * correction_factor
)

In [53]:
correction_factor = (
    y_test["Target_PM10"].mean()
    /
    pred_pm10.mean()
)

adjusted_pred_pm10 = (
    pred_pm10 * correction_factor
)

## Evaluation Metrics

In [54]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### PM2.5 Model

In [55]:
print(
    "PM25 MAE:",
    mean_absolute_error(
        y_test["Target_PM25"],
        adjusted_pred_pm25
    )
)


print(
    "PM25 RMSE:",
    np.sqrt(
        mean_squared_error(
            y_test["Target_PM25"],
            adjusted_pred_pm25
        )
    )
)


print(
    "PM25 R2:",
    r2_score(
        y_test["Target_PM25"],
        adjusted_pred_pm25
    )
)

PM25 MAE: 13.513366021439776
PM25 RMSE: 25.902225779081434
PM25 R2: 0.6866007640361265


## PM10 Model

In [57]:
print(
    "PM10 MAE:",
    mean_absolute_error(
        y_test["Target_PM10"],
        adjusted_pred_pm10
    )
)


print(
    "PM10 RMSE:",
    np.sqrt(
        mean_squared_error(
            y_test["Target_PM10"],
            adjusted_pred_pm10
        )
    )
)


print(
    "PM10 R2:",
    r2_score(
        y_test["Target_PM10"],
        adjusted_pred_pm10
    )
)

PM10 MAE: 69.2743396515471
PM10 RMSE: 154.91293848846232
PM10 R2: 0.43426134434562447


### Evaluation by City

In [58]:
test_results = final_dataset.loc[X_test.index].copy()

test_results["Predicted_PM25"] = adjusted_pred_pm25
test_results["Predicted_PM10"] = adjusted_pred_pm10


for city in test_results["City"].unique():

    city_data = test_results[
        test_results["City"] == city
    ]

    print("\n", city)

    print(
        "PM25 MAE:",
        mean_absolute_error(
            city_data["Target_PM25"],
            city_data["Predicted_PM25"]
        )
    )

    print(
        "PM10 MAE:",
        mean_absolute_error(
            city_data["Target_PM10"],
            city_data["Predicted_PM10"]
        )
    )


 Bengaluru
PM25 MAE: 29.09326523693952
PM10 MAE: 92.84072691980181


In [59]:
print(final_dataset.shape)
print(X.shape)
print(len(X_test))

print(final_dataset.index[:5])
print(X_test.index[:5])

(6480, 34)
(6480, 31)
1296
Index([0, 1, 2, 3, 4], dtype='int64')
RangeIndex(start=0, stop=5, step=1)


## Feature Importance

In [60]:
importance = pd.DataFrame(
    {
        "Feature": X_train.columns,
        "Importance": model_pm25.feature_importances_
    }
)


importance = importance.sort_values(
    "Importance",
    ascending=False
)


importance.head(15)

,Feature,Importance
29,City_Delhi,0.971227
15,PM25_lag_1,0.006443
20,PM10_lag_1,0.003487
25,PM25_roll_6,0.002453
13,month,0.001643
12,day,0.001542
14,day_of_week,0.001174
9,SO2,0.000985
26,PM25_roll_24,0.000955
3,Pressure,0.000953


In [61]:
importance_pm10 = pd.DataFrame(
    {
        "Feature": X_train.columns,
        "Importance": model_pm10.feature_importances_
    }
)

importance_pm10.sort_values(
    "Importance",
    ascending=False
).head(15)

,Feature,Importance
27,PM10_roll_6,0.649189
29,City_Delhi,0.143958
6,WindDirection,0.020750
13,month,0.019084
20,PM10_lag_1,0.018273
1,Humidity,0.016816
12,day,0.014566
14,day_of_week,0.012579
26,PM25_roll_24,0.010934
25,PM25_roll_6,0.009583


## Correlation

In [62]:
final_dataset[
    [
        "PM25",
        "PM10",
        "PM25_lag_1",
        "PM10_lag_1",
        "Target_PM25"
    ]
].corr()

,PM25,PM10,PM25_lag_1,PM10_lag_1,Target_PM25
PM25,1.000000,0.932736,0.986140,0.926676,0.837767
PM10,0.932736,1.000000,0.911355,0.980047,0.759688
PM25_lag_1,0.986140,0.911355,1.000000,0.932781,0.831198
PM10_lag_1,0.926676,0.980047,0.932781,1.000000,0.758208
Target_PM25,0.837767,0.759688,0.831198,0.758208,1.000000


## Actual V/S Predicted

In [66]:
comparison = pd.DataFrame({
    "Actual": y_test['Target_PM10'],
    "Predicted": adjusted_pred_pm10
})

comparison.head(5)

,Actual,Predicted
0,16.5,15.605284
1,16.1,16.392980
2,15.5,16.593821
3,14.9,16.765986
4,14.6,16.765986


In [67]:
comparison = pd.DataFrame({
    "Actual": y_test['Target_PM25'],
    "Predicted": adjusted_pred_pm25
})

comparison.head(5)

,Actual,Predicted
0,13.1,12.349660
1,12.4,14.571447
2,11.6,13.855298
3,10.9,13.923107
4,10.4,14.040400


## Save model, csv, and feature columns

In [68]:
final_dataset.to_csv("Delhi_Mumbai_BLR_final_dataset.csv", index=False)

In [69]:
import joblib

In [70]:
joblib.dump(model_pm25, "pm25_model.pkl")
joblib.dump(model_pm10, "pm10_model.pkl")

['pm10_model.pkl']

In [71]:
joblib.dump(X_train.columns.tolist(), "feature_columns.pkl")

['feature_columns.pkl']

In [72]:
PM25_metadata = {
    "target": "Target_PM25",
    "model": "XGBoost",
    "train_size": len(X_train),
    "test_size": len(X_test),
    "PM25_MAE": 13.513,
    "PM25_RMSE": 25.902,
    "PM25_R2": 0.686
}
joblib.dump(PM25_metadata, "PM25_metadata.pkl")

['PM25_metadata.pkl']

In [73]:
PM10_metadata = {
    "target": "Target_PM10",
    "model": "XGBoost",
    "train_size": len(X_train),
    "test_size": len(X_test),
    "PM10_MAE": 69.274,
    "PM10_RMSE": 154.912,
    "PM10_R2": 0.434
}
joblib.dump(PM10_metadata, "PM10_metadata.pkl")

['PM10_metadata.pkl']